In [34]:
from nemo.collections.asr.parts.submodules.wfst_decoder import RivaGpuWfstDecoder

from inference_funcs import load_bit_phoneme_model, evaluate_model
from dataset import getDatasetLoaders
import numpy as np
import torch
import torch.nn.functional as F

langugage_model_fst_path = "/data/code/nejm-brain-to-text/language_model/pretrained_language_models/openwebtext_1gram_lm_sil/TLG_with_symbols.fst"

decoder = RivaGpuWfstDecoder(lm_fst=langugage_model_fst_path, decoding_mode="nbest", 
                             beam_size=18, lm_weight=1.0, nbest_size=18, max_mem=400000000, blank_penalty=0.70)

In [3]:
device = 'cuda'

bit_phoneme_filepath = "/data/models/transformer_short_training_fixed_seed_0/"
model, args = load_bit_phoneme_model(bit_phoneme_filepath)
model = model.to(device)

data_file = '/data/neural_data/ptDecoder_ctc_both'
trainLoaders, testLoaders, loadedData = getDatasetLoaders(
        data_file, 8, None, 
        False
    )

outputs, cer, per_day_cer = evaluate_model(model, loadedData, args, partition='test', device='cuda', verbose=False)

num_classes = 41
logits = np.zeros((len(outputs['logits']), max(outputs['logitLengths']), num_classes))
for idx, l in enumerate(outputs['logits']):
    l_length = outputs['logitLengths'][idx]
    logits[idx, :l_length, :] = l
    
logits_torch = torch.from_numpy(logits)
log_probs = F.log_softmax(logits_torch, dim=-1).to(dtype=torch.float32, device=device)
log_probs_length = torch.from_numpy(np.array(outputs['logitLengths'])).to(dtype=torch.int64, device='cpu')

In [4]:
log_probs_blank_last = torch.concat((log_probs[:, :, 1:], log_probs[:, :, 0:1]), dim=-1) # move blank to end
log_probs_arranged = torch.concat((log_probs_blank_last[:, :, -1:], log_probs_blank_last[:, :, -2:-1], log_probs_blank_last[:, :, :-2]), dim=-1)

In [6]:
hypotheses = decoder._decode_nbest(log_probs_arranged, log_probs_length)

LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
LOG ([5.5]:RebuildRepository():lat/determinize-lattice-pruned.cc:287) Rebuilding repository.
WARNING ([5.5]:CheckMemoryUsage():lat/determinize-lattice-pruned.cc:320) Did not reach requested beam in determinize-lattice: size exceeds maximum 400000000 bytes; (repo,arcs,elems) = (261414208,3976064,138325824), after rebuilding, repo size was 197033408, effective beam was 17.1798 vs. requested beam 18
LOG ([5.5]:RebuildRepository():lat/determi

In [10]:
idx = 400
print(outputs['transcriptions'][idx])
for i in range(18):
    print(hypotheses[idx]._hypotheses[i])

friday afternoon at five thirty
WfstNbestUnit(words=('VOID', 'AFFERENT', 'AT', 'FIVE', 'LADY'), timesteps=(0, 11, 21, 31, 38), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 27, 10, 10, 10, 1, 1, 1, 3, 15, 15, 13, 4, 24, 24, 32, 1, 1, 0, 3, 32, 32, 1, 1, 15, 15, 7, 36, 36, 1, 1, 22, 22, 22, 14, 10, 19, 1, 1, 1, 1, 1), score=83141.109375)
WfstNbestUnit(words=('VOID', 'AFFERENT', 'AT', 'FIVE', 'CLAYEY'), timesteps=(0, 11, 21, 31, 38), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 27, 10, 10, 10, 1, 1, 1, 3, 15, 15, 13, 4, 24, 24, 32, 1, 1, 0, 3, 32, 32, 1, 1, 15, 15, 7, 36, 36, 1, 1, 21, 22, 22, 14, 14, 19, 1, 1, 1, 1, 1), score=84199.921875)
WfstNbestUnit(words=('VOID', 'AFFERENT', 'AT', 'FIVE', 'PLAY'), timesteps=(0, 11, 21, 31, 38), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 27, 10, 10, 10, 1, 1, 1, 3, 15, 15, 13, 4, 24, 24, 32, 1, 1, 0, 3, 32, 32, 1, 1, 15, 15, 7, 36, 36, 1, 1, 28, 22, 22, 14, 14, 14, 1, 1, 1, 1, 1), score=84459.0625)
WfstNbestUnit(words=('VOID', 'AFFERENT', 'AT', 'FIVE', 'LEYH'), timesteps=

In [32]:
# First second only (10 frames = 1s if each frame = 100ms)
end = min(17, log_probs_arranged.shape[1])  # safety check in case seq is shorter

log_probs_chunk = log_probs_arranged[:, 0:10, :]  # take first 10 frames
log_probs_len_chunk = torch.tensor([end] * log_probs_arranged.shape[0])  # length per batch

# Decode just this first-second chunk
hypotheses_first_second = decoder._decode_nbest(log_probs_chunk, log_probs_len_chunk)


In [33]:
idx = 400
print(outputs['transcriptions'][idx])
for i in range(18):
    print(hypotheses_first_second[idx]._hypotheses[i])


friday afternoon at five thirty
WfstNbestUnit(words=('VOID',), timesteps=(0,), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 27, 10, 10, 10, 1, 1, 1, 0, 0), score=37705.78125)
WfstNbestUnit(words=('VOID', 'F'), timesteps=(0, 11), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 27, 10, 10, 10, 1, 1, 1, 12, 15), score=39267.18359375)
WfstNbestUnit(words=('VOID', 'PFAFF'), timesteps=(0, 11), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 27, 10, 10, 10, 1, 1, 15, 3, 15), score=39870.66015625)
WfstNbestUnit(words=('VOID', 'DEAF'), timesteps=(0, 11), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 27, 10, 10, 10, 1, 1, 10, 12, 15), score=41441.09375)
WfstNbestUnit(words=('VOID', 'EV'), timesteps=(0, 11), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 27, 10, 10, 10, 1, 1, 1, 12, 36), score=42034.734375)
WfstNbestUnit(words=('VOID', 'IF'), timesteps=(0, 11), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 27, 10, 10, 10, 1, 1, 1, 18, 15), score=42386.7578125)
WfstNbestUnit(words=('VOID', 'DEV'), timesteps=(0, 11), alignment=(0, 0, 0, 0, 0, 0, 36, 27, 2